# 固定原价整股五会话HOLD/SOXL/SOXX全分钟标签

事前冻一次Qlib读取、来源支持门和一次Recorder。先只用已完成分钟与下一bar真实原价/费用核固定股数的全年资金/整股支持，不读取终端收益；通过后对全部合法意向生成HOLD与两资产五会话自身状态路径。2021/2022价值封存、不用于本卡特征或方向选择。

In [1]:
import os,json,hashlib,warnings,time
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.core.recorder_run import publish_evidence_manifest
from src.analysis.stage1_fixed_share_lock_batch import scan_fixed_share_source,build_fixed_share_lock_labels
root=Path.cwd();card=root/'research/stage1_v5_fixed_share_lock_labels_20261030';raw_plan=(card/'plan.json').read_bytes();plan=json.loads(raw_plan)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256']
assert sha(root/plan['source_support_card']/'summary.json')==plan['source_support_summary_sha256'] and sha(root/plan['source_support_card']/'perday_support.csv')==plan['source_support_perday_sha256']
for key in ('fixed_order_code','vector_code'):
 assert sha(root/plan[key])==plan[key+'_sha256']
assert sha(root/plan['mechanical_parity_card']/'audit.json')==plan['mechanical_parity_audit_sha256'] and sha(root/plan['tariff_plan'])==plan['tariff_plan_sha256']
for role in ('parent','frame'):
 s=plan[role];source=root/s['card']
 assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256']
 assert json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
book=read('parent','analysis/native/dated_f696_0')
frame=read('frame','inputs/covered_frame').reset_index(drop=True)
perday_old=pd.read_csv(root/plan['source_support_card']/'perday_support.csv',parse_dates=['trade_date','session_close'])
assert len(frame)==len(perday_old)==plan['expected_sessions'] and book['audit']['status']=='passed' and book['audit']['slippage_bps']==0
tariff=json.loads((root/plan['tariff_plan']).read_text())['tariff']
print('verified source identities',len(frame))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[34087:MainThread](2026-10-30 14:56:30,288) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[34087:MainThread](2026-10-30 14:56:30,290) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[34087:MainThread](2026-10-30 14:56:30,290) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


verified source identities 1056


In [2]:
started=time.perf_counter()
raw=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$raw_close','$price_valid'],start_time=plan['start']+' 09:31:00',end_time=plan['end']+' 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=raw.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+f.removeprefix('$') for f in x.columns];parts.append(x)
assert parts[0].index.equals(parts[1].index)
quotes=pd.concat(parts,axis=1).sort_index();assert len(raw)==820440 and quotes.index.is_unique
print('Qlib minute rows',len(raw))

Qlib minute rows 820440


In [3]:
# Source/whole-share gate comes before any five-session economic value.
fixed_perday,source=scan_fixed_share_source(frame,book,quotes,perday_old,tariff=tariff)
assert source['intent_rows']==plan['expected_decision_clock_cash_intents'] and source['mature_path_rows']==plan['expected_mature_path_source_intents']
years={x['year']:x for x in source['annual']};gate=plan['support_gate']
discovery=sum(years[y]['paired_funded_days'] for y in (2018,2019,2020))
passed=bool(discovery>=gate['minimum_discovery_paired_funded_days_2018_2020'] and all(years[y]['paired_funded_days']>=gate['minimum_'+str(y)+'_paired_funded_days'] for y in (2019,2020,2021,2022)))
source_gate={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'support_code_sha256':sha(root/'src/analysis/stage1_fixed_share_lock_batch.py'),'source':source,'gate_pass':passed,'future_value_columns_accessed':False}
(card/'source_gate.json').write_text(json.dumps(source_gate,ensure_ascii=False,indent=2)+'\n')
if not passed:raise ValueError('blocked: fixed-share whole-share source support gate failed')
print(json.dumps({'gate_pass':passed,'annual':source['annual'],'paired_funded_rows':source['paired_funded_rows']},ensure_ascii=False))

{"gate_pass": true, "annual": [{"year": 2018, "days": 53, "paired_funded_days": 53, "paired_funded_rows": 19091, "mature_path_rows": 19091, "decision_no_whole_share": 0, "decision_cash_insufficient": 0, "insufficient_fill_cash": 0}, {"year": 2019, "days": 252, "paired_funded_days": 252, "paired_funded_rows": 91944, "mature_path_rows": 91944, "decision_no_whole_share": 0, "decision_cash_insufficient": 0, "insufficient_fill_cash": 0}, {"year": 2020, "days": 253, "paired_funded_days": 249, "paired_funded_rows": 91023, "mature_path_rows": 91023, "decision_no_whole_share": 0, "decision_cash_insufficient": 0, "insufficient_fill_cash": 0}, {"year": 2021, "days": 252, "paired_funded_days": 252, "paired_funded_rows": 92304, "mature_path_rows": 92304, "decision_no_whole_share": 0, "decision_cash_insufficient": 0, "insufficient_fill_cash": 0}, {"year": 2022, "days": 246, "paired_funded_days": 242, "paired_funded_rows": 88634, "mature_path_rows": 88634, "decision_no_whole_share": 0, "decision_cash

In [4]:
labels,mechanics=build_fixed_share_lock_labels(frame,book,quotes,perday_old,fixed_perday,tariff=tariff,maximum_intent_rows=plan['budget']['maximum_intent_rows'],maximum_seconds=plan['budget']['maximum_seconds'])
assert len(labels)==mechanics['intent_rows']==source['intent_rows']
assert mechanics['mature_path_rows']==source['mature_path_rows'] and mechanics['paired_funded_rows']==source['paired_funded_rows']
assert labels.duplicated(['trade_date','bar_end_at']).sum()==0
mature=labels.status.isin(['paired_funded','attempt_no_fill'])
assert labels.loc[mature,'label_available_at'].gt(labels.loc[mature,'decision_at']).all()
assert labels.loc[labels.status.eq('attempt_no_fill'),['soxl_value','soxx_value']].notna().all().all()
# Only discovery2018-2020 economic columns are accessed below.
discovery_labels=labels.loc[labels.trade_date.dt.year.le(2020)&mature,['trade_date','relative_value_soxl_minus_soxx','soxl_tail_damage','soxx_tail_damage']].copy()
byday=discovery_labels.groupby('trade_date').agg(value_mean=('relative_value_soxl_minus_soxx','mean'),any_positive=('relative_value_soxl_minus_soxx',lambda x:bool(x.gt(0).any())),any_negative=('relative_value_soxl_minus_soxx',lambda x:bool(x.lt(0).any())),soxl_tail_mean=('soxl_tail_damage','mean'),soxx_tail_mean=('soxx_tail_damage','mean')).reset_index();byday['year']=byday.trade_date.dt.year
annual=byday.groupby('year').agg(days=('trade_date','size'),positive_days=('any_positive','sum'),negative_days=('any_negative','sum'),day_mean_relative_value=('value_mean','mean'),day_mean_soxl_tail_damage=('soxl_tail_mean','mean'),day_mean_soxx_tail_damage=('soxx_tail_mean','mean')).reset_index()
annual_list=[{'year':int(x.year),'days':int(x.days),'positive_days':int(x.positive_days),'negative_days':int(x.negative_days),'day_mean_relative_value':float(x.day_mean_relative_value),'day_mean_soxl_tail_damage':float(x.day_mean_soxl_tail_damage),'day_mean_soxx_tail_damage':float(x.day_mean_soxx_tail_damage)} for x in annual.itertuples(index=False)]
sign=plan['discovery_sign_gate'];year_map={x['year']:x for x in annual_list}
sign_pass=bool(all(year_map[y]['positive_days']>=sign[f'minimum_{y}_positive_days'] and year_map[y]['negative_days']>=sign[f'minimum_{y}_negative_days'] for y in (2019,2020)))
elapsed=time.perf_counter()-started
if elapsed>plan['budget']['maximum_seconds']:raise ValueError('blocked: source plus label wall time budget exceeded')
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'batch_code_sha256':sha(root/'src/analysis/stage1_fixed_share_lock_batch.py'),'order_code_sha256':plan['fixed_order_code_sha256'],'vector_code_sha256':plan['vector_code_sha256'],'source':source,'source_gate_pass':passed,'mechanics':mechanics,'discovery_annual':annual_list,'discovery_sign_gate_pass':sign_pass,'full_pre_recorder_elapsed_seconds':elapsed,'held_2021_2022_value_columns_inspected':False,'historical_research_passed':False,'decision':'fixed_share_five_session_labels_source_and_sign_pass_information_unproven' if sign_pass else 'fixed_share_five_session_labels_sign_support_failed'}
print(json.dumps(audit,ensure_ascii=False,indent=2))

fixed_label_progress 0 367 0.4


fixed_label_progress 100 36707 36.1


fixed_label_progress 200 73227 72.9


fixed_label_progress 300 109567 109.8


fixed_label_progress 400 146267 146.6


...

fixed_label_progress 700 256007 263.4


fixed_label_progress 800 292527 303.3


fixed_label_progress 900 329227 344.3


fixed_label_progress 1000 365927 385.5


{
  "experiment": "stage1_v5_fixed_share_lock_labels_20261030",
  "plan_sha256": "d1b792cb7f52b926a37614ce3ac41b6089c3c76b27cd6d0eb3f5817e8924dfee",
  "batch_code_sha256": "5242a360d60281f1d80d3fa500aadd2fea6b5995a4d8a5097d02212cab0f2f86",
  "order_code_sha256": "5bf69f1d17e266d869ce13ba6511ea8b1675b1b24e3d382fe4285f20d3deefe7",
  "vector_code_sha256": "c813c720e56891ad5c75bf38ea454ad32c5615449fbc63644fe18a1c20b21f32",
  "source": {
    "annual": [
      {
        "year": 2018,
        "days": 53,
        "paired_funded_days": 53,
        "paired_funded_rows": 19091,
        "mature_path_rows": 19091,
        "decision_no_whole_share": 0,
        "decision_cash_insufficient": 0,
        "insufficient_fill_cash": 0
      },
      {
        "year": 2019,
        "days": 252,
        "paired_funded_days": 252,
        "paired_funded_rows": 91944,
        "mature_path_rows": 91944,
        "decision_no_whole_share": 0,
        "decision_cash_insufficient": 0,
        "insufficient_fill_cas

In [5]:
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 recorder=R.get_recorder()
 recorder.save_objects(artifact_path='inputs',plan=plan,source_gate=source_gate)
 recorder.save_objects(artifact_path='sources',batch_builder=(root/'src/analysis/stage1_fixed_share_lock_batch.py').read_text(),fixed_order=(root/plan['fixed_order_code']).read_text(),vectorized_path=(root/plan['vector_code']).read_text())
 recorder.save_objects(artifact_path='analysis',labels=labels,source_perday=fixed_perday,discovery_annual=annual,audit=audit)
 recorder.log_metrics(intent_rows=float(source['intent_rows']),mature_path_rows=float(source['mature_path_rows']),paired_funded_rows=float(source['paired_funded_rows']),source_gate_pass=1.,discovery_sign_gate_pass=float(sign_pass))
assert recorder.status=='FINISHED'
local=tracking/str(recorder.experiment_id)/recorder.id
recorder_size=sum(x.stat().st_size for x in local.rglob('*') if x.is_file())
if recorder_size>plan['budget']['maximum_recorder_bytes']:raise ValueError('blocked: Recorder exceeds frozen size; preserve local')
summary={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'decision':audit['decision'],'historical_research_passed':False,'recorder_id':recorder.id,'experiment_id':str(recorder.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(local),'recorder_bytes':recorder_size,'source_gate':source_gate,'mechanics':mechanics,'discovery_annual':annual_list,'new_model_fits':0,'new_native_candidate_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']},'limitations':['Five-session local fixed-raw-share event/HOLD labels, not a trained continuous allocator or full SOXX candidate account.','2021/2022 values stored but not inspected for selection; overlapping minutes require day-grouping and five-session purge.','Actual broker rejection/partial fill, first receipt and separate dividend receivables remain unknown.']}
(card/'workflow.yaml').write_text('experiment: '+card.name+'\nnotebook: '+card.name+'.ipynb\nrecorder_uri: '+str(tracking)+'\nartifact_policy: slim_card_recorder_first\n')
publish_evidence_manifest(card,summary)
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'recorder_id':recorder.id,'rows':len(labels),'paired_funded':source['paired_funded_rows'],'recorder_bytes':recorder_size},ensure_ascii=False))

[34087:MainThread](2026-10-30 15:05:56,812) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_v5_fixed_share_lock_labels_20261030.


[34087:MainThread](2026-10-30 15:05:56,813) INFO - qlib.workflow - [exp.py:258] - Experiment 670393730550395432 starts running ...


[34087:MainThread](2026-10-30 15:05:56,907) INFO - qlib.workflow - [recorder.py:345] - Recorder b85e80ab42fa4d9c83c6c9667114fd5d starts running under Experiment 670393730550395432 ...


[34087:MainThread](2026-10-30 15:05:58,197) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"recorder_id": "b85e80ab42fa4d9c83c6c9667114fd5d", "rows": 385932, "paired_funded": 382996, "recorder_bytes": 71670789}


## 中文结论

SOXX硬门未通过；0新模型/候选账户。Qlib Recorder `b85e80ab42fa4d9c83c6c9667114fd5d` 保存385932事前分钟意向、382996成熟五会话路径；新固定股数动作在全部成熟路径两臂各自一笔整股订单，旧完整版多单混杂已消除。父本峰值误差4.44e−16，566秒批次/71.7MB Recorder在预算内，NAS档案及四件选定对象远端哈希读回通过。2018–2020按日等权SOXL−SOXX局部均值−1.477/−0.436/−1.042bp，费用差+0.009/+0.009/+0.006bp，价格与持仓路径主导负值；与旧交集仅约+0.008至+0.013bp改善，新增旧多单事件在2020年均值−1.679bp。正负日支持不等于预测力。2021/2022价值封存未选特征。下一步做分钟自身回撤相对开盘状态/普通分钟价量的匹配信息门，再考虑连续目标和SOXX全账户。详见`conclusion.md`与归因JSON。